# ADLS Scope and Authentication Overview
This notebook checks available secret scopes, validates access to the retail ADLS credentials, and configures Spark to authenticate to the storage account.

In [0]:
# Purpose: List all secret scopes available in the workspace so the correct retail ADLS scope can be identified.
dbutils.secrets.listScopes()

In [0]:
# Purpose: Inspect the legacy retail scope to confirm whether it contains any secrets.
dbutils.secrets.list("retail-adls-scope")

In [0]:
# Purpose: Validate that the service principal client ID can be retrieved from the active key vault-backed scope.
dbutils.secrets.get(
    scope="retail-adls-kv-scope-v2",
    key="adls-sp-client-id"
)

In [0]:
# Purpose: Load the storage account name and service principal secrets required for ADLS OAuth authentication.
storage_account_name = "silveradlsstorage"

tenant_id = dbutils.secrets.get(
    scope="retail-adls-kv-scope-v2",
    key="adls-sp-tenant-id"
)

client_id = dbutils.secrets.get(
    scope="retail-adls-kv-scope-v2",
    key="adls-sp-client-id"
)

client_secret = dbutils.secrets.get(
    scope="retail-adls-kv-scope-v2",
    key="adls-sp-client-secret"
)

print("Secrets Loaded Successfully")

In [0]:
# Purpose: Apply OAuth Spark configuration so this cluster can read and write files in the retail ADLS Gen2 storage account.
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net",
    client_id
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net",
    client_secret
)

spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("ADLS Authentication Configured")